In [0]:
%pip install openpyxl
%pip install xgboost
%pip install tqdm_joblib
%pip install seaborn
%pip install shap
import shap
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import ElasticNet, LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, ParameterGrid
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, r2_score, matthews_corrcoef, make_scorer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

import seaborn as sns
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import sys


In [0]:
# Load all the relevant CDS files
itrax_xover_10y = pd.read_excel('/dbfs/FileStore/tables/itrax_xover_10y.xlsx')
itrax_xover_5y = pd.read_excel('/dbfs/FileStore/tables/itrax_xover_5y.xlsx')
itrax_gen_10y = pd.read_excel('/dbfs/FileStore/tables/itrax_gen_10y.xlsx')
itrax_gen_5y = pd.read_excel('/dbfs/FileStore/tables/itrax_gen_5y.xlsx')
cdx_gen_10y = pd.read_excel('/dbfs/FileStore/tables/cdx_gen_10y.xlsx')

# Convert to datetime and calculate the forward change
cds_list = [itrax_xover_10y, itrax_xover_5y, itrax_gen_10y, itrax_gen_5y, cdx_gen_10y]
for df in cds_list:
  df['Date'] = pd.to_datetime(df['Date']).dt.normalize()
  #------------------- Sanity Check -------------------
  #df['Forward Change'] = df['Last Price'] - df['Last Price'].shift(1)
  #df['Change (1)'] = 1
  #-----------------------------------------------------
  df['Forward Change'] =df['Last Price'].shift(-1) - df['Last Price']
  df['Change (1)'] = df['Last Price'] - df['Last Price'].shift(1) # Lag 1 of the Return i.e. how much it was from yesterday to today 
  df['Change (2)'] = df['Last Price'].shift(1) - df['Last Price'].shift(2) # Lag 2 of the Return i.e. how much it was from 2 days ago to yesterday
  df['Change (3)'] = df['Last Price'].shift(2) - df['Last Price'].shift(3)
  df['Change (4)'] = df['Last Price'].shift(3) - df['Last Price'].shift(4)
  df.dropna(inplace = True)

# Load the file with the features for the American CDS and calculate some more features we will use
american_features = pd.read_excel('/dbfs/FileStore/tables/cds_features.xlsx')
american_features['date'] = pd.to_datetime(american_features['date']).dt.normalize()
american_features['2-10Y Slope'] = (american_features['dgs2'] - american_features['dgs10']) / 8 # Slope of the yield curve between 2Y and 10 Y
american_features['10-30Y Slope'] = (american_features['dgs10'] - american_features['dgs30']) / 20 # Slope of the yield curve between 10Y and 30Y
american_features['Aaa - 10Y Premium'] = american_features['daaa'] - american_features['dgs10'] # Premium of Moody's Aaa Yield on top of 10Y Treasuries
american_features['Baa - 10Y Premium'] = american_features['dbaa'] - american_features['dgs10'] # Premium of Moody's Baa Yield on top of 10Y Treasuries
american_features.dropna(inplace = True) # drop all the NA values i.e. weekends which were outputted as features




In [0]:
'''
# Load the different indexes that will be used for training, validation and/or testing
df_train_us_PI = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.Index_PI_US_Daily_W").toPandas()
df_val_us_PI = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.index_df_val_PI_US").toPandas()

df_train_us_SR = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.index_df_train_SR_US_2024_06_01").toPandas()

# Create and X and y matrixes for training and validation
Xy_train_US = pd.merge(df_train_us_PI, cdx_gen_10y, left_on='period', right_on='Date', how='left') # Merge Index and CDS Data
Xy_train_US = pd.merge(Xy_train_US, american_features, left_on='period', right_on='date', how='left') # Merge Index, CDS and Feature Data
Xy_train_US = Xy_train_US[['period', 'index_raw', 'index_ema','sum_weighted_sentiment','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)', 'Forward Change']] # Subset only for all the X variables + Forward CHange (y variable) + the date variable Period

Xy_train_US.dropna(inplace = True) # Drop all cases where the y variable 'Forward Change' is missing
Xy_train_US.reset_index(inplace = True, drop = True) # Reset the index so that everything counts from 0


X_train = Xy_train_US[['index_raw', 'index_ema','sum_weighted_sentiment','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)']] # Isolate the features we will be using
#X_train = Xy_train_US[['Change (1)', 'Baa - 10Y Premium', 'dgs2', 'dgs10']]

y_train = Xy_train_US[['Forward Change']] # y variable for regression
y_train['Positive'] = y_train['Forward Change'] > 0 # y variable for classification

Xy_val_US = pd.merge(df_val_us_PI, cdx_gen_10y, left_on='period', right_on='Date', how='left') # Merge Index and CDS Data
Xy_val_US = pd.merge(Xy_val_US, american_features, left_on='period', right_on='date', how='left') # Merge Index, CDS and Feature Data
Xy_val_US = Xy_val_US[['period', 'index_raw', 'index_ema','sum_weighted_sentiment', '2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)', 'Forward Change']] # Subset only for all the X variables + Forward CHange (y variable)
Xy_val_US.dropna(inplace = True) # Drop all cases where the y variable 'Forward Change' is missing
Xy_val_US.reset_index(inplace = True, drop = True) # Reset the index so that everything counts from 0

X_val = Xy_val_US[['index_raw', 'index_ema','sum_weighted_sentiment','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)']] # Isolate the features we will be using
#X_val = Xy_val_US[['Change (1)', 'Baa - 10Y Premium', 'dgs2', 'dgs10']]

y_val = Xy_val_US[['Forward Change']] # y variable for regression
y_val['Positive'] = y_val['Forward Change'] > 0 # y variable for classification
'''

In [ ]:
df_PI = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.Index_PI_US_Daily_W").toPandas()
df_PU = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.Index_PU_US_Daily_W").toPandas()
#df_SR = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.Index_SR_US_Daily_W").toPandas()

Xy = pd.merge(df_PI, cdx_gen_10y, left_on='period', right_on='Date', how='left', suffixes = ('_PI', '')) # Merge Index and CDS Data
Xy = pd.merge(Xy, df_PU, left_on='period', right_on='period', how='left', suffixes = ('', '_PU')) # Merge Index and CDS Data
#Xy = pd.merge(Xy, df_SR, left_on='period', right_on='period', how='left', suffixes = ('', '_SR')) # Merge Index and CDS Data
Xy = pd.merge(Xy, american_features, left_on='period', right_on='date', how='left') # Merge Index, CDS and Feature Data

Xy = Xy[['period', 'index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)', 'Forward Change']]
#Xy = Xy[['period', 'index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI', 'index_raw_SR', 'index_ema_SR','sum_weighted_sentiment_SR','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)', 'Forward Change']]

X = Xy[['period', 'index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)']]
y = Xy[['Forward Change']] # y variable for regression
y['Positive'] = y['Forward Change'] > 0 # y variable for classification

X_train = X[X['period'] < '2024-01-01'].drop(columns = ['period']) # Isolate the features we will be using for training
y_train = y[X['period'] < '2024-01-01'].drop(columns = ['period']) # Isolate the y variable for training
X_val = X[(X['period'] >= '2024-01-01') & (X['period'] < '2025-01-01')].drop(columns = ['period']) # Isolate the features we will be using for validation
y_val = y[(y['period'] >= '2024-01-01') & (y['period'] < '2025-01-01')].drop(columns = ['period']) # Isolate the y variable for validation

X_test = X[(X['period'] >= '2025-01-01') & (X['period'] < '2025-02-01')].drop(columns = ['period']) # Isolate the features we will be using for testing
y_test = y[(y['period'] >= '2025-01-01') & (y['period'] < '2025-02-01')].drop(columns = ['period']) # Isolate the y variable for testing

X_train_no_index = X_train.drop(columns = ['index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI']) # Isolate the features we will be using for training without the index variables
X_val_no_index = X_val.drop(columns = ['index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI']) # Isolate the features we will be using for validation without the index variables
X_test_no_index = X_test.drop(columns = ['index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI']) # Isolate the features we will be using for testing without the index variables

X_train_index = X_train[['index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI']] # Isolate the index features for training
X_val_index = X_val[['index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI']] # Isolate the index features for validation
X_test_index = X_test[['index_raw_PU', 'index_ema_PU','sum_weighted_sentiment_PU', 'index_raw_PI', 'index_ema_PI','sum_weighted_sentiment_PI']] # Isolate the index features for testing


In [0]:
'''
#--------------------------------------------------------------------------------------------------------
#------------------------------Experimental Code mutiple indexes----------------------------------------
#--------------------------------------------------------------------------------------------------------

# Load the different indexes that will be used for training, validation and/or testing
df_train_us_PI = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.index_df_train_PI_DE").toPandas()
df_val_us_PI = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.index_df_val_PI_DE").toPandas()

df_train_us_SR = spark.table("hive_metastore.nax279xq3_ds_academy_qfin_lab_2025.index_df_train_SR_DE_2024_06_01").toPandas()

# Create and X and y matrixes for training and validation
Xy_train_US = pd.merge(df_train_us_PI, itrax_xover_5y, left_on='period', right_on='Date', how='left') # Merge Index and CDS Data
Xy_train_US = pd.merge(Xy_train_US, american_features, left_on='period', right_on='date', how='left') # Merge Index, CDS and Feature Data
Xy_train_US = Xy_train_US[['period', 'index_raw', 'index_ema','sum_weighted_sentiment','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)', 'Forward Change']] # Subset only for all the X variables + Forward CHange (y variable) + the date variable Period
Xy_train_US = pd.merge(Xy_train_US, df_train_us_SR[['period', 'index_raw', 'index_ema', 'sum_weighted_sentiment']], left_on='period', right_on='period', how='left', suffixes = ('', '_SR')) # Merge Index, CDS, Features and SR Data

Xy_train_US.dropna(inplace = True) # Drop all cases where the y variable 'Forward Change' is missing
Xy_train_US.reset_index(inplace = True, drop = True) # Reset the index so that everything counts from 0


X_train = Xy_train_US[['index_raw', 'index_ema','sum_weighted_sentiment', 'index_raw_SR', 'index_ema_SR','sum_weighted_sentiment_SR','2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)']] # Isolate the features we will be using
#X_train = Xy_train_US[['Change (1)', 'Baa - 10Y Premium', 'dgs2', 'dgs10']]

y_train = Xy_train_US[['Forward Change']] # y variable for regression
y_train['Positive'] = y_train['Forward Change'] > 0 # y variable for classification


Xy_val_US = pd.merge(df_val_us_PI, itrax_xover_5y, left_on='period', right_on='Date', how='left') # Merge Index and CDS Data
Xy_val_US = pd.merge(Xy_val_US, american_features, left_on='period', right_on='date', how='left') # Merge Index, CDS and Feature Data
Xy_val_US = Xy_val_US[['period', 'index_raw', 'index_ema','sum_weighted_sentiment', '2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)', 'Forward Change']] # Subset only for all the X variables + Forward CHange (y variable)

display(Xy_val_US)
Xy_val_US = pd.merge(Xy_val_US, df_train_us_SR[['period', 'index_raw', 'index_ema', 'sum_weighted_sentiment']], left_on='period', right_on='period', how='left', suffixes = ('', '_SR')) # Merge Index, CDS, Features and SR Data


Xy_val_US.dropna(inplace = True) # Drop all cases where the y variable 'Forward Change' is missing
Xy_val_US.reset_index(inplace = True, drop = True) # Reset the index so that everything counts from 0

X_val = Xy_val_US[['index_raw', 'index_ema','sum_weighted_sentiment', 'index_raw_SR', 'index_ema_SR','sum_weighted_sentiment_SR', '2-10Y Slope', '10-30Y Slope', 'Aaa - 10Y Premium', 'Baa - 10Y Premium', 'dgs2','dgs10', 'Change (1)', 'Change (2)', 'Change (3)', 'Change (4)']] # Isolate the features we will be using
#X_val = Xy_val_US[['Change (1)', 'Baa - 10Y Premium', 'dgs2', 'dgs10']]

y_val = Xy_val_US[['Forward Change']] # y variable for regression
y_val['Positive'] = y_val['Forward Change'] > 0 # y variable for classification
'''

In [0]:
def optimize_xgb_classifier(model, X_train, y_train, K_folds = 5): # A function that will be used perform Hyperparameter optimization via a K Fold Grid-search for the XGB Classifier
    param_grid = {'max_depth': [3, 6, 9, 12, 15], 'eta': [0.005, 0.05, 0.1, 0.2], 'gamma': [0, 0.01, 0.1, 0.5],  'subsample': [0.75, 1], 'lambda': [0.5, 1, 2], 'n_estimators': [50, 200, 500]}


    mcc_scorer = make_scorer(matthews_corrcoef)
    cv = StratifiedKFold(n_splits=K_folds, shuffle=False) # By default we split into 10 folds
    grid = GridSearchCV( estimator=model, param_grid=param_grid, scoring=mcc_scorer, cv=cv, n_jobs=-1, verbose=2, refit = True)
    grid.fit(X_train, y_train)

    print("Best Matthews Correlation Coefficient:", grid.best_score_)
    print("Best params:", grid.best_params_)
    best_model = grid.best_estimator_

    return best_model




In [0]:

##############################################################################################################################
# Models for predictions using the Indexes + Macro Economic Features
##############################################################################################################################
# Define the base models we will fit
model_regression_xgb = XGBRegressor(eta = 0.05, n_estimators = 50)
model_classification_xgb = XGBClassifier(eta = 0.005, n_estimators = 50, objective = 'binary:logistic', eval_metric = 'auc', gamma = 0, reg_lambda = 1, max_depth = 9, subsample = 1)
model_regressor_rf = RandomForestRegressor(n_estimators = 50, max_depth = 10, min_samples_leaf = 2, min_samples_split = 4, max_features = 'sqrt', bootstrap = False)
model_classification_rf = RandomForestClassifier(n_estimators = 50, max_depth = 10, min_samples_leaf = 2, min_samples_split = 4, max_features = 'sqrt', bootstrap = False)

#model_classification_xgb = optimize_xgb_classifier(model_classification_xgb, X_train, y_train['Positive'], 2)

# Fit the models on the training data
model_regression_xgb.fit(X_train, y_train['Forward Change'])
model_classification_xgb.fit(X_train, y_train['Positive'])
model_regressor_rf.fit(X_train, y_train['Forward Change'])
model_classification_rf.fit(X_train, y_train['Positive'])

# Get predictions for the regression models
y_val_pred_reg_xgb = model_regression_xgb.predict(X_val)
y_val_pred_reg_rf = model_regressor_rf.predict(X_val)
# Get predictions for the classfier models
y_val_pred_class_xgb = model_classification_xgb.predict(X_val)
y_val_pred_class_rf = model_classification_rf.predict(X_val)


########################################################################################################################
# Models for predictions using only the Macro Economic Features (i.e. without the Indexes)
########################################################################################################################
model_regression_xgb_no_index = XGBRegressor(eta = 0.05, n_estimators = 50)
model_classification_xgb_no_index = XGBClassifier(eta = 0.005, n_estimators = 50, objective = 'binary:logistic', eval_metric = 'auc', gamma = 0, reg_lambda = 1, max_depth = 9, subsample = 1)
model_regressor_rf_no_index = RandomForestRegressor(n_estimators = 50, max_depth = 10, min_samples_leaf = 2, min_samples_split = 4, max_features = 'sqrt', bootstrap = False)
model_classification_rf_no_index = RandomForestClassifier(n_estimators = 50, max_depth = 10, min_samples_leaf = 2, min_samples_split = 4, max_features = 'sqrt', bootstrap = False)

#model_classification_xgb_no_index = optimize_xgb_classifier(model_classification_xgb_no_index, X_train_no_index, y_train['Positive'], 2)

# Fit the models on the training data
model_regression_xgb_no_index.fit(X_train_no_index, y_train['Forward Change'])
model_classification_xgb_no_index.fit(X_train_no_index, y_train['Positive'])
model_regressor_rf_no_index.fit(X_train_no_index, y_train['Forward Change'])
model_classification_rf_no_index.fit(X_train_no_index, y_train['Positive'])

# Get predictions for the regression models
y_val_pred_reg_xgb_no_index = model_regression_xgb_no_index.predict(X_val_no_index)
y_val_pred_reg_rf_no_index = model_regressor_rf_no_index.predict(X_val_no_index)
# Get predictions for the classfier models
y_val_pred_class_xgb_no_index = model_classification_xgb_no_index.predict(X_val_no_index)
y_val_pred_class_rf_no_index = model_classification_rf_no_index.predict(X_val_no_index)

########################################################################################################################
# Model for predictions using only the Indexes (i.e. without the Macro Economic Features)
########################################################################################################################
model_regression_xgb_index = XGBRegressor(eta = 0.05, n_estimators = 50)
model_classification_xgb_index = XGBClassifier(eta = 0.005, n_estimators = 50, objective = 'binary:logistic', eval_metric = 'auc', gamma = 0, reg_lambda = 1, max_depth = 9, subsample = 1)
model_regressor_rf_index = RandomForestRegressor(n_estimators = 50, max_depth = 10, min_samples_leaf = 2, min_samples_split = 4, max_features = 'sqrt', bootstrap = False)
model_classification_rf_index = RandomForestClassifier(n_estimators = 50, max_depth = 10, min_samples_leaf = 2, min_samples_split = 4, max_features = 'sqrt', bootstrap = False)

model_regression_xgb_index.fit(X_train_index, y_train['Forward Change'])
model_classification_xgb_index.fit(X_train_index, y_train['Positive'])
model_regressor_rf_index.fit(X_train_index, y_train['Forward Change'])
model_classification_rf_index.fit(X_train_index, y_train['Positive'])

# Get predictions for the regression models
y_val_pred_reg_xgb_index = model_regression_xgb_index.predict(X_val_index)
y_val_pred_reg_rf_index = model_regressor_rf_index.predict(X_val_index)
# Get predictions for the classfier models
y_val_pred_class_xgb_index = model_classification_xgb_index.predict(X_val_index)
y_val_pred_class_rf_index = model_classification_rf_index.predict(X_val_index)



In [0]:
##################################################################################################
# Predictions for the Index + Macro Economic Features Models
##################################################################################################
# Confusion Matrixes for the Classifier Models

# Confusion Matrixes for the Classifier Models
cm_xgb_val = confusion_matrix(y_val['Positive'], y_val_pred_class_xgb)
xgb_val_mcc = matthews_corrcoef(y_val['Positive'], y_val_pred_class_xgb)
disp_xgb_val = ConfusionMatrixDisplay(confusion_matrix=cm_xgb_val, display_labels=model_classification_xgb.classes_)
disp_xgb_val.plot(cmap='Blues', values_format='d')  # use '.2f' for normalized
plt.title(f"Confusion Matrix XG Boost Val, MCC: {xgb_val_mcc:.2f}")
plt.show()

cm_rf_val = confusion_matrix(y_val['Positive'], y_val_pred_class_rf)
rf_val_mcc = matthews_corrcoef(y_val['Positive'], y_val_pred_class_rf)
disp_rf_val = ConfusionMatrixDisplay(confusion_matrix=cm_rf_val, display_labels=model_classification_rf.classes_)
disp_rf_val.plot(cmap='Blues', values_format='d')
plt.title(f"Confusion Matrix RF Val, MCC: {rf_val_mcc:.2f}")
plt.show()

In [ ]:
##################################################################################################
# Predictions for Macro Economic Features Models
##################################################################################################

# Confusion Matrixes for the Classifier Models
cm_xgb_val_no_index = confusion_matrix(y_val['Positive'], y_val_pred_class_xgb_no_index)
xgb_val_mcc_no_index = matthews_corrcoef(y_val['Positive'], y_val_pred_class_xgb_no_index)
disp_xgb_val_no_index = ConfusionMatrixDisplay(confusion_matrix=cm_xgb_val_no_index, display_labels=model_classification_xgb_no_index.classes_)
disp_xgb_val_no_index.plot(cmap='Blues', values_format='d')  # use '.2f' for normalized
plt.title(f"Confusion Matrix XG Boost Val, MCC: {xgb_val_mcc_no_index:.2f}")
plt.show()

cm_rf_val_no_index = confusion_matrix(y_val['Positive'], y_val_pred_class_rf_no_index)
rf_val_mcc_no_index = matthews_corrcoef(y_val['Positive'], y_val_pred_class_rf_no_index)
disp_rf_val_no_index = ConfusionMatrixDisplay(confusion_matrix=cm_rf_val_no_index, display_labels=model_classification_rf_no_index.classes_)
disp_rf_val_no_index.plot(cmap='Blues', values_format='d')
plt.title(f"Confusion Matrix RF Val, MCC: {rf_val_mcc_no_index:.2f}")
plt.show()

In [ ]:
##################################################################################################
# Predictions for Index Models
##################################################################################################
# Confusion Matrixes for the Classifier Models
cm_xgb_val_index = confusion_matrix(y_val['Positive'], y_val_pred_class_xgb_no_index)
xgb_val_mcc_index = matthews_corrcoef(y_val['Positive'], y_val_pred_class_xgb_no_index)
disp_xgb_val_index = ConfusionMatrixDisplay(confusion_matrix=cm_xgb_val_index, display_labels=model_classification_xgb_no_index.classes_)
disp_xgb_val_index.plot(cmap='Blues', values_format='d')  # use '.2f' for normalized
plt.title(f"Confusion Matrix XG Boost Val, MCC: {xgb_val_mcc_index:.2f}")
plt.show()

cm_rf_val_index = confusion_matrix(y_val['Positive'], y_val_pred_class_rf_no_index)
rf_val_mcc_no_index = matthews_corrcoef(y_val['Positive'], y_val_pred_class_rf_no_index)
disp_rf_val_index = ConfusionMatrixDisplay(confusion_matrix=cm_rf_val_index, display_labels=model_classification_rf_no_index.classes_)
disp_rf_val_index.plot(cmap='Blues', values_format='d')
plt.title(f"Confusion Matrix RF Val, MCC: {rf_val_mcc_no_index:.2f}")
plt.show()

In [0]:
import numpy as np

plt.plot(y_val['Forward Change'][0:100], label = 'True')
plt.plot(y_val_pred_reg_xgb[0:100], label = 'Predicted XGB')
plt.title('Predicted vs True XGB')
plt.show()


plt.plot(y_val['Forward Change'][0:100], label = 'True')
plt.plot(y_val_pred_reg_rf[0:100], label = 'Predicted RF')
plt.title('Predicted vs True RF')
plt.legend()
plt.show()

plt.plot(y_val['Forward Change'].cumsum(), label = 'CDS Realized Change')
plt.plot(np.cumsum(y_val_pred_reg_xgb), label = 'XGB Predicted Change')
plt.plot(np.cumsum(y_val_pred_reg_rf), label = 'RF Predicted Change')
plt.legend()
plt.show()



In [0]:
explainer_xgboost = shap.TreeExplainer(model_regression_xgb)
shap_values_xgboost = explainer_xgboost.shap_values(X_train)
shap.summary_plot(shap_values_xgboost, X_train)



In [ ]:
# Plot a correlation matrix of the features and y
corr = Xy[[col for col in Xy.columns if col != 'period']].corr(numeric_only = True, method = 'pearson')
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix Features and y')
plt.show()